#### Sentinel-2 滇池区域数据下载


In [1]:
import ee
import geemap
ee.Authenticate(auth_mode='localhost')
ee.Initialize(project='earth-engine-auth-project')   


/Users/luo/miniconda3/envs/ygp_water/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


#### 参数设置

In [ ]:
# ── 研究区域：滇池 ────
region = ee.Geometry.Rectangle([102.55, 24.6, 102.82, 25.03])

# ── 数据集与波段 ─────
dset     = 'COPERNICUS/S2_SR_HARMONIZED'
band_sel   = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']  # Blue,Green,Red,NIR,SWIR1,SWIR2
band_vis   = ['B4', 'B3', 'B2']                      # 真彩色

print(f'研究区面积: {region.area().getInfo() / 1e6:.1f} km²')  # type: ignore



研究区面积: 1302954811.2477183 m²


In [ ]:
## sentinel-2
img_col = (ee.ImageCollection(dset)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
           .filterBounds(region)
           .filterDate('2025-07-01', '2025-12-31')
           .sort('CLOUDY_PIXEL_PERCENTAGE')           
           )
img_col


In [ ]:
img_col = img_col.filter(ee.Filter.contains('.geo', region))  
img_sel = img_col.first().clip(region).select(band_sel) 
img_sel


In [ ]:
Map = geemap.Map()
Map.centerObject(region, 6)
Map.addLayer(img_sel, {'bands': band_vis, 'min': 0, 'max': 2000}, \
             'Sentinel-2', True, 1)
Map



Map(center=[24.814936237058983, 102.68500000000006], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# type: ignore
task = ee.batch.Export.image.toDrive(   
    image=img_sel,
    description='s2_20251206',
    folder='gee-down',
    scale=20,     
    fileFormat='GeoTIFF',
    region=region)
task.start()
print('Export task started:', task.status())  



Export task started: {'state': 'READY', 'description': 's2_20251206', 'priority': 100, 'creation_timestamp_ms': 1781237604796, 'update_timestamp_ms': 1781237604796, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'ZVW7HMSMP6ELPRTYVSZ2QGVN', 'name': 'projects/earth-engine-auth-project/operations/ZVW7HMSMP6ELPRTYVSZ2QGVN'}
